# Notebook 4a — Training: Gaussian Noise

> **To create 4b (Uniform) or 4c (Laplace):** duplicate this notebook and change only the `NOISE_TYPE` variable in cell 3. Everything else stays identical.

This notebook trains the U-Net to predict Gaussian noise and saves:
- Checkpoints every N epochs → `checkpoints/gaussian/`
- Training loss log → `logs/gaussian_loss.csv`
- Sample grid at end → `samples/gaussian_final.png`

In [ ]:
!pip install -q torch torchvision tqdm matplotlib numpy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import math, os, csv, time
from tqdm.notebook import tqdm

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/diffusion_noise_project'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## ⚡ Only change this cell when duplicating for 4b / 4c

In [ ]:
# ============================================================
# CHANGE THIS for 4b → 'uniform', for 4c → 'laplace'
NOISE_TYPE = 'gaussian'
# ============================================================

assert NOISE_TYPE in ('gaussian', 'uniform', 'laplace'), f'Unknown noise type: {NOISE_TYPE}'
print(f'Training with: {NOISE_TYPE.upper()} noise')

## 1. Config (identical across all three notebooks)

In [ ]:
CONFIG = {
    'dataset': 'MNIST', 'image_size': 28, 'channels': 1, 'batch_size': 128,
    'T': 1000, 'beta_start': 1e-4, 'beta_end': 0.02, 'schedule': 'linear',
    'num_epochs': 30, 'lr': 2e-4, 'optimizer': 'adamw', 'weight_decay': 1e-4,
    'num_workers': 2, 'model_channels': 64, 'channel_mults': (1, 2, 4),
    'log_every': 100, 'save_every': 5, 'seed': 42,
}

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_seed(CONFIG['seed'])

# Directories
ckpt_dir = os.path.join(PROJECT_DIR, 'checkpoints', NOISE_TYPE)
os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(os.path.join(PROJECT_DIR, 'logs'), exist_ok=True)
os.makedirs(os.path.join(PROJECT_DIR, 'samples'), exist_ok=True)

LOG_PATH = os.path.join(PROJECT_DIR, 'logs', f'{NOISE_TYPE}_loss.csv')
print(f'Checkpoint dir: {ckpt_dir}')
print(f'Log path: {LOG_PATH}')

## 2. Beta Schedule

In [ ]:
def make_beta_schedule(schedule, T, beta_start, beta_end):
    if schedule == 'linear':
        betas = torch.linspace(beta_start, beta_end, T)
    elif schedule == 'cosine':
        steps = T + 1
        x = torch.linspace(0, T, steps)
        alphas_cumprod = torch.cos(((x / T) + 0.008) / 1.008 * torch.pi / 2) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        betas = betas.clamp(0, 0.999)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    alphas_cumprod_prev = torch.cat([torch.tensor([1.0]), alphas_cumprod[:-1]])
    return {
        'betas': betas, 'alphas': alphas,
        'alphas_cumprod': alphas_cumprod,
        'alphas_cumprod_prev': alphas_cumprod_prev,
        'sqrt_alphas_cumprod': alphas_cumprod.sqrt(),
        'sqrt_one_minus_alphas_cumprod': (1 - alphas_cumprod).sqrt(),
    }

schedule = make_beta_schedule(CONFIG['schedule'], CONFIG['T'], CONFIG['beta_start'], CONFIG['beta_end'])

## 3. Noise Distributions

In [ ]:
class ForwardDiffusion:
    def __init__(self, schedule, device):
        self.device = device
        self.T = len(schedule['betas'])
        for k, v in schedule.items():
            setattr(self, k, v.to(device))
    def sample_noise(self, shape):
        raise NotImplementedError
    def sample_timestep(self, batch_size):
        return torch.randint(0, self.T, (batch_size,), device=self.device).long()
    def q_sample(self, x_0, t):
        eps = self.sample_noise(x_0.shape)
        sqrt_ab = self.sqrt_alphas_cumprod[t][:, None, None, None]
        sqrt_1ab = self.sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
        return sqrt_ab * x_0 + sqrt_1ab * eps, eps

class GaussianDiffusion(ForwardDiffusion):
    def sample_noise(self, shape):
        return torch.randn(shape, device=self.device)

class UniformDiffusion(ForwardDiffusion):
    _a = 3 ** 0.5
    def sample_noise(self, shape):
        return torch.empty(shape, device=self.device).uniform_(-self._a, self._a)

class LaplaceDiffusion(ForwardDiffusion):
    _b = 2 ** -0.5
    def sample_noise(self, shape):
        u = torch.empty(shape, device=self.device).uniform_(-0.5, 0.5)
        return -self._b * u.sign() * torch.log(1 - 2 * u.abs())

DIFFUSION_CLASSES = {
    'gaussian': GaussianDiffusion,
    'uniform':  UniformDiffusion,
    'laplace':  LaplaceDiffusion,
}

diffusion = DIFFUSION_CLASSES[NOISE_TYPE](schedule, DEVICE)
print(f'Diffusion process: {NOISE_TYPE}')

## 4. U-Net Model

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.proj = nn.Sequential(nn.Linear(dim, dim*4), nn.SiLU(), nn.Linear(dim*4, dim*4))
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / (half-1))
        args = t[:,None].float() * freqs[None]
        return self.proj(torch.cat([torch.sin(args), torch.cos(args)], dim=-1))

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, num_groups=8):
        super().__init__()
        self.norm1 = nn.GroupNorm(min(num_groups, in_ch), in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(min(num_groups, out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.act = nn.SiLU()
        self.time_proj = nn.Linear(time_emb_dim, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t_emb):
        h = self.act(self.norm1(x))
        h = self.conv1(h)
        h = h + self.time_proj(self.act(t_emb))[:,:,None,None]
        h = self.act(self.norm2(h))
        h = self.conv2(h)
        return h + self.skip(x)

class UNet(nn.Module):
    def __init__(self, in_channels=1, model_channels=64, channel_mults=(1,2,4)):
        super().__init__()
        time_emb_dim = model_channels * 4
        self.time_emb = SinusoidalTimeEmbedding(model_channels)
        chs = [model_channels * m for m in channel_mults]
        self.input_conv = nn.Conv2d(in_channels, chs[0], 3, padding=1)
        self.enc1 = ResBlock(chs[0], chs[0], time_emb_dim)
        self.down1 = nn.Conv2d(chs[0], chs[0], 4, stride=2, padding=1)
        self.enc2 = ResBlock(chs[0], chs[1], time_emb_dim)
        self.down2 = nn.Conv2d(chs[1], chs[1], 4, stride=2, padding=1)
        self.enc3 = ResBlock(chs[1], chs[2], time_emb_dim)
        self.mid1 = ResBlock(chs[2], chs[2], time_emb_dim)
        self.mid2 = ResBlock(chs[2], chs[2], time_emb_dim)
        self.up3 = nn.ConvTranspose2d(chs[2], chs[2], 4, stride=2, padding=1)
        self.dec3 = ResBlock(chs[2]+chs[1], chs[1], time_emb_dim)
        self.up2 = nn.ConvTranspose2d(chs[1], chs[1], 4, stride=2, padding=1)
        self.dec2 = ResBlock(chs[1]+chs[0], chs[0], time_emb_dim)
        self.dec1 = ResBlock(chs[0]+chs[0], chs[0], time_emb_dim)
        self.out_norm = nn.GroupNorm(8, chs[0])
        self.out_act = nn.SiLU()
        self.out_conv = nn.Conv2d(chs[0], in_channels, 1)
    def forward(self, x, t):
        t_emb = self.time_emb(t)
        h0 = self.input_conv(x)
        h1 = self.enc1(h0, t_emb); h1d = self.down1(h1)
        h2 = self.enc2(h1d, t_emb); h2d = self.down2(h2)
        h3 = self.enc3(h2d, t_emb)
        h = self.mid2(self.mid1(h3, t_emb), t_emb)
        h = F.interpolate(self.up3(h), size=h2.shape[2:], mode='nearest')
        h = self.dec3(torch.cat([h, h2], dim=1), t_emb)
        h = F.interpolate(self.up2(h), size=h1.shape[2:], mode='nearest')
        h = self.dec2(torch.cat([h, h1], dim=1), t_emb)
        h = self.dec1(torch.cat([h, h0], dim=1), t_emb)
        return self.out_conv(self.out_act(self.out_norm(h)))

model = UNet(
    in_channels=CONFIG['channels'],
    model_channels=CONFIG['model_channels'],
    channel_mults=CONFIG['channel_mults']
).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {trainable:,}')

## 5. Data

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.MNIST(
    root='/content/data', train=True, download=True, transform=transform
)
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=CONFIG['batch_size'],
    shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=True
)

print(f'Training samples: {len(train_dataset):,}')
print(f'Batches per epoch: {len(train_loader)}')

## 6. Resume from Checkpoint (if available)

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay']
)
scaler = torch.cuda.amp.GradScaler()  # mixed precision

start_epoch = 0
all_losses = []  # list of (global_step, loss)
epoch_losses = []

# Look for latest checkpoint
ckpts = sorted([f for f in os.listdir(ckpt_dir) if f.endswith('.pt')])
if ckpts:
    latest = os.path.join(ckpt_dir, ckpts[-1])
    ckpt = torch.load(latest, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch = ckpt['epoch'] + 1
    all_losses = ckpt.get('all_losses', [])
    epoch_losses = ckpt.get('epoch_losses', [])
    print(f'Resumed from {latest} (epoch {ckpt["epoch"]})')
else:
    print('Starting fresh.')

## 7. Training Loop

In [ ]:
# Initialise CSV log
if not os.path.exists(LOG_PATH):
    with open(LOG_PATH, 'w', newline='') as f:
        csv.writer(f).writerow(['epoch', 'step', 'loss', 'epoch_avg_loss', 'elapsed_s'])

global_step = start_epoch * len(train_loader)
t0 = time.time()

for epoch in range(start_epoch, CONFIG['num_epochs']):
    model.train()
    running_loss = 0.0
    n_batches = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{CONFIG["num_epochs"]} [{NOISE_TYPE}]')

    for x_0, _ in pbar:
        x_0 = x_0.to(DEVICE)
        t = diffusion.sample_timestep(x_0.shape[0])

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            x_t, eps = diffusion.q_sample(x_0, t)
            eps_pred = model(x_t, t)
            loss = F.mse_loss(eps_pred, eps)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        loss_val = loss.item()
        running_loss += loss_val
        n_batches += 1
        global_step += 1
        all_losses.append((global_step, loss_val))

        if global_step % CONFIG['log_every'] == 0:
            pbar.set_postfix({'loss': f'{loss_val:.4f}'})

    avg_loss = running_loss / n_batches
    epoch_losses.append(avg_loss)
    elapsed = time.time() - t0

    # Append to CSV
    with open(LOG_PATH, 'a', newline='') as f:
        csv.writer(f).writerow([epoch+1, global_step, loss_val, avg_loss, f'{elapsed:.1f}'])

    print(f'Epoch {epoch+1:3d} | avg loss: {avg_loss:.5f} | elapsed: {elapsed/60:.1f} min')

    # Save checkpoint
    if (epoch + 1) % CONFIG['save_every'] == 0 or (epoch + 1) == CONFIG['num_epochs']:
        ckpt_path = os.path.join(ckpt_dir, f'epoch_{epoch+1:03d}.pt')
        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'loss': avg_loss,
            'all_losses': all_losses,
            'epoch_losses': epoch_losses,
            'noise_type': NOISE_TYPE,
            'config': CONFIG,
        }, ckpt_path)
        print(f'  Checkpoint saved: {ckpt_path}')

print('\nTraining complete!')

## 8. Training Loss Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per-step loss (smoothed)
steps, losses = zip(*all_losses)
window = 50
smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
axes[0].plot(steps[:len(smoothed)], smoothed, alpha=0.9, color='steelblue')
axes[0].set_title(f'{NOISE_TYPE.capitalize()} — Step loss (smoothed, window={window})')
axes[0].set_xlabel('Training step')
axes[0].set_ylabel('MSE loss')
axes[0].grid(True, alpha=0.3)

# Epoch average loss
axes[1].plot(range(1, len(epoch_losses)+1), epoch_losses, marker='o', color='steelblue')
axes[1].set_title(f'{NOISE_TYPE.capitalize()} — Epoch average loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Avg MSE loss')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'figures', f'{NOISE_TYPE}_loss.png'), dpi=150)
plt.show()

## 9. Generate Samples

In [ ]:
@torch.no_grad()
def ddpm_sample(model, diffusion, n_samples=16, img_size=28, channels=1, device=DEVICE):
    model.eval()
    x = torch.randn((n_samples, channels, img_size, img_size), device=device)
    for t_val in tqdm(reversed(range(diffusion.T)), total=diffusion.T, desc='Sampling'):
        t_batch = torch.full((n_samples,), t_val, device=device, dtype=torch.long)
        eps_pred = model(x, t_batch)
        alpha_t     = diffusion.alphas[t_val]
        alpha_bar_t = diffusion.alphas_cumprod[t_val]
        beta_t      = diffusion.betas[t_val]
        coeff = (1 - alpha_t) / (1 - alpha_bar_t).sqrt()
        mean  = (1 / alpha_t.sqrt()) * (x - coeff * eps_pred)
        x = mean + (beta_t.sqrt() * torch.randn_like(x) if t_val > 0 else 0)
    return x.clamp(-1, 1)

print('Generating samples (this takes ~2 min on T4)...')
samples = ddpm_sample(model, diffusion, n_samples=64)

# Display
samples_display = (samples * 0.5 + 0.5).clamp(0, 1)
grid = torchvision.utils.make_grid(samples_display, nrow=8, padding=2)

fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
ax.axis('off')
ax.set_title(f'Generated samples — {NOISE_TYPE.capitalize()} diffusion', fontsize=14)
plt.tight_layout()
save_path = os.path.join(PROJECT_DIR, 'samples', f'{NOISE_TYPE}_final.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Samples saved to {save_path}')

In [ ]:
print(f'Notebook 4 ({NOISE_TYPE}) complete.')
print('Artefacts saved:')
print(f'  Checkpoints: {ckpt_dir}/')
print(f'  Loss log:    {LOG_PATH}')
print(f'  Samples:     {save_path}')
print('Proceed to Notebooks 4b/4c, then Notebook 5 for evaluation.')